# 3D Cube Animation Using Rotation Matrices

This notebook demonstrates how to construct and animate a **3D cube** using Python, NumPy, and Matplotlib.  
The animation is based on **linear algebra transformations**, specifically **rotation matrices**.

---

# 1. Mathematical Representation of a Cube

A cube has **8 vertices**.

We define a cube centered at the origin with coordinates:

Bottom square (z = -1)

(-1, -1, -1)  
( 1, -1, -1)  
( 1,  1, -1)  
(-1,  1, -1)

Top square (z = 1)

(-1, -1,  1)  
( 1, -1,  1)  
( 1,  1,  1)  
(-1,  1,  1)

---

# 2. Vertex Matrix Representation

Instead of storing vertices separately, we store them in a **matrix form**.

Each column represents **one vertex**.

\[
P =
\begin{bmatrix}
-1 & 1 & 1 & -1 & -1 & 1 & 1 & -1 \\
-1 & -1 & 1 & 1 & -1 & -1 & 1 & 1 \\
-1 & -1 & -1 & -1 & 1 & 1 & 1 & 1
\end{bmatrix}
\]

Interpretation:

| Index | Vertex |
|------|------|
| 0 | (-1,-1,-1) |
| 1 | (1,-1,-1) |
| 2 | (1,1,-1) |
| 3 | (-1,1,-1) |
| 4 | (-1,-1,1) |
| 5 | (1,-1,1) |
| 6 | (1,1,1) |
| 7 | (-1,1,1) |

---

# 3. Cube Edges

Edges define **which vertices are connected by lines**.

A cube has **12 edges**.

Bottom face:

(0,1), (1,2), (2,3), (3,0)

Top face:

(4,5), (5,6), (6,7), (7,4)

Vertical edges:

(0,4), (1,5), (2,6), (3,7)

These pairs represent **connections between vertices**.

---

# 4. Why Use Vertex Indices Instead of Coordinates?

Edges are stored as **pairs of vertex indices**, not coordinates.

Example:

(0,1)

means:

connect vertex 0 → vertex 1.

This approach is powerful because:

- vertices can move (during rotation)
- edges automatically update
- geometry structure remains constant

This is the same idea used in **3D mesh models**.

---

# 5. Rotation of the Cube

Rotation in 3D space is achieved using **rotation matrices**.

Rotation around the X-axis:

\[
$R_x(\theta)$=
\begin{bmatrix}
1 & 0 & 0 \\
0 & \cos\theta & -\sin\theta \\
0 & \sin\theta & \cos\theta
\end{bmatrix}
\]

Rotation around the Z-axis:

\[
$R_z (\theta)$=
\begin{bmatrix}
\cos\theta & -\sin\theta & 0 \\
\sin\theta & \cos\theta & 0 \\
0 & 0 & 1
\end{bmatrix}
\]

---

# 6. Rotating the Cube

The vertex matrix is rotated using matrix multiplication:

\[
P' = R P
\]

where:

- \(P\) = original vertex matrix
- \(R\) = rotation matrix
- \(P'\) = rotated vertices

Because the matrix multiplication is applied to the **entire matrix**, all vertices rotate **simultaneously**.

---

# Summary

A 3D object can be represented as:

Vertices → positions of points  
Edges → connections between points  

Animation is created by applying **time-dependent transformations** to the vertex matrix.

\[
P'(t) = R(t)P
\]


## Cube

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
plt.rcParams["animation.embed_limit"] = 50  # MB


def base_setup():
    fig, ax = plt.subplots(
        figsize=(6, 6),
        dpi=100,
        subplot_kw={"projection": "3d"}
    )

    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_zlim(-1.5, 1.5)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    ax.set_box_aspect([1,1,1])
    ax.view_init(elev=25, azim=30)

    fig.patch.set_facecolor("lightgray") #whole canvas
    ax.set_facecolor("cyan") #background of 3D boxes
    ax.xaxis.pane.set_facecolor("lightblue") #background of x-axis plane
    ax.yaxis.pane.set_facecolor("lightgreen") #background of y-axis plane
    ax.zaxis.pane.set_facecolor("lightyellow") #background of z-axis plane
    ax.grid(False) #turn off grid

    return fig, ax

In [ ]:
fig, ax = base_setup()

# cube vertices
P = np.array([
[-1, 1, 1,-1,-1, 1, 1,-1],
[-1,-1, 1, 1,-1,-1, 1, 1],
[-1,-1,-1,-1, 1, 1, 1, 1]
])


# cube edges
edges = [
(0,1),(1,2),(2,3),(3,0),
(4,5),(5,6),(6,7),(7,4),
(0,4),(1,5),(2,6),(3,7)
]

lines = []
for e in edges:
    line, = ax.plot(
        [P[0,e[0]], P[0,e[1]]],
        [P[1,e[0]], P[1,e[1]]],
        [P[2,e[0]], P[2,e[1]]],
        color='blue'
    )
    lines.append(line)

def Rx(t):
    return np.array([
        [1,0,0],
        [0,np.cos(t),-np.sin(t)],
        [0,np.sin(t), np.cos(t)]
    ])

def Rz(t):
    return np.array([
        [np.cos(t),-np.sin(t),0],
        [np.sin(t), np.cos(t),0],
        [0,0,1]
    ])

def update(frame):

    theta = frame * 0.04

    R = Rz(theta) @ Rx(theta)
    P_rot = R @ P

    for line,edge in zip(lines,edges):
        i,j = edge
        line.set_data(
            [P_rot[0,i],P_rot[0,j]],
            [P_rot[1,i],P_rot[1,j]]
        )

        line.set_3d_properties(
            [P_rot[2,i],P_rot[2,j]]
        )

        ax.set_title(f"rigid body rotation (cube)\nangle={theta:.2f} rad")

    return lines

ani = FuncAnimation(fig, update, frames=200, interval=60, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

#ave the animation as an MP4 file
# ani.save(
#     "/home/shubh/projects/3d-anim/assets/videos/cube_rotation.mp4",
#     writer="ffmpeg",
#     fps=24,
#     dpi=200,
#     bitrate=2400
# )

## Sphere

In [ ]:
fig, ax = base_setup()

# --------------------------------
# SPHERE PARAMETERS
# --------------------------------
r = 1 # radius of sphere

THETA = np.linspace(0, np.pi, 30) # polar angle (0 to pi)
PHI = np.linspace(0, 2*np.pi, 30) # azimuthal angle (0 to 2pi)

Theta, Phi = np.meshgrid(THETA, PHI)

# --------------------------------
# SPHERICAL → CARTESIAN
# --------------------------------
X = r * np.sin(Theta) * np.cos(Phi)
Y = r * np.sin(Theta) * np.sin(Phi)
Z = r * np.cos(Theta)

# --------------------------------
# ROTATION MATRIX (Z axis)
# --------------------------------
def rotate_z(x, y, z, angle):

    cos = np.cos(angle)
    sin = np.sin(angle)

    x_new = cos*x - sin*y
    y_new = sin*x + cos*y
    z_new = z

    return x_new, y_new, z_new

def rotate_x(x, y, z, angle):
    
    cos = np.cos(angle)
    sin = np.sin(angle)

    x_new = x
    y_new = cos*y - sin*z
    z_new = sin*y + cos*z

    return x_new, y_new, z_new

def rotate_y(x, y, z, angle):
    
    cos = np.cos(angle)
    sin = np.sin(angle)

    x_new = cos*x + sin*z
    y_new = y
    z_new = -sin*x + cos*z

    return x_new, y_new, z_new

def update(frame):

    ax.clear()

    ax.set_xlim(-1.5,1.5)
    ax.set_ylim(-1.5,1.5)
    ax.set_zlim(-1.5,1.5)

    ax.grid(False) #turn off grid

    ax.set_xlabel("X") 
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    ax.set_box_aspect([1,1,1])

    angle = frame * 0.05

    Xr, Yr, Zr = rotate_y(X, Y, Z, angle)

    ax.plot_wireframe(Xr, Yr, Zr, color="magenta")
    ax.set_title(f"rigid body rotation (sphere)\nangle={angle:.2f} rad")

    
ani = FuncAnimation(
    fig,
    update,
    frames=200,
    interval=50
)

plt.close(fig)
HTML(ani.to_jshtml())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# SPHERE PARAMETERS
r = 1

theta = np.linspace(0, np.pi, 30)
phi = np.linspace(0, 2*np.pi, 30)

Theta, Phi = np.meshgrid(theta, phi)

# spherical → cartesian
X = r * np.sin(Theta) * np.cos(Phi)
Y = r * np.sin(Theta) * np.sin(Phi)
Z = r * np.cos(Theta)

# rotation functions
def rotate_x(x, y, z, angle):

    c = np.cos(angle)
    s = np.sin(angle)

    x_new = x
    y_new = c*y - s*z
    z_new = s*y + c*z

    return x_new, y_new, z_new


def rotate_y(x, y, z, angle):

    c = np.cos(angle)
    s = np.sin(angle)

    x_new = c*x + s*z
    y_new = y
    z_new = -s*x + c*z

    return x_new, y_new, z_new


def rotate_z(x, y, z, angle):

    c = np.cos(angle)
    s = np.sin(angle)

    x_new = c*x - s*y
    y_new = s*x + c*y
    z_new = z

    return x_new, y_new, z_new


# --------------------------------
# FIGURE SETUP (3 SUBPLOTS)
# --------------------------------
fig = plt.figure(figsize=(12,4))

ax1 = fig.add_subplot(131, projection="3d")
ax2 = fig.add_subplot(132, projection="3d")
ax3 = fig.add_subplot(133, projection="3d")


def setup_axis(ax):
    ax.set_xlim(-1.5,1.5)
    ax.set_ylim(-1.5,1.5)
    ax.set_zlim(-1.5,1.5)

    ax.set_box_aspect([1,1,1])
    ax.grid(False)
    ax.view_init(elev=25, azim=30)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    fig.patch.set_facecolor("lightgray") #whole canvas
    ax.set_facecolor("cyan") #background of 3D boxes
    ax.xaxis.pane.set_facecolor("lightblue") #background of x-axis plane
    ax.yaxis.pane.set_facecolor("lightgreen") #background of y-axis plane
    ax.zaxis.pane.set_facecolor("lightyellow") #background of z-axis plane


def update(frame):

    ax1.clear()
    ax2.clear()
    ax3.clear()

    setup_axis(ax1)
    setup_axis(ax2)
    setup_axis(ax3)

    angle = frame * 0.05

    # X rotation
    Xx, Yx, Zx = rotate_x(X, Y, Z, angle)
    ax1.plot_wireframe(Xx, Yx, Zx, color="red")
    ax1.set_title("Rotation about X-axis")

    # Y rotation
    Xy, Yy, Zy = rotate_y(X, Y, Z, angle)
    ax2.plot_wireframe(Xy, Yy, Zy, color="green")
    ax2.set_title("Rotation about Y-axis")

    # Z rotation
    Xz, Yz, Zz = rotate_z(X, Y, Z, angle)
    ax3.plot_wireframe(Xz, Yz, Zz, color="blue")
    ax3.set_title("Rotation about Z-axis")


ani = FuncAnimation(
    fig,
    update,
    frames=200,
    interval=50
)

plt.close(fig)
HTML(ani.to_jshtml())

# ani.save(
#     "/home/shubh/projects/3d-anim/assets/gifs/sphere_multiple_axes_rotation.gif",
#     writer="ffmpeg",
#     fps=24,
#     dpi=200,
#     bitrate=2400
# )